# Scraper-Researcher with Apache Fluss durable storage

A research agent that scrapes targets, **persists the knowledge corpus to an Apache Fluss
cluster**, and answers questions with Claude. Because the corpus lives in Fluss (a streaming
storage system), a fresh process re-hydrates its search index straight from Fluss — no
re-scraping. Embeddings via **Ollama**, answers via **Claude**.

## Prereqs
- `mvn -DskipTests package`
- **Ollama**: `podman exec ollama ollama pull nomic-embed-text`
- **Fluss cluster**: `podman compose -f docker-compose-fluss.yml up -d` (coordinator+tablet+zk)
- Answers: `export ANTHROPIC_API_KEY=sk-ant-...`
- `pip install -e python/`

## 1. Bootstrap

In [1]:
import os, pathlib, subprocess, time
repo_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
if 'AGENTIC_FLINK_JAR' not in os.environ:
    jars = [j for j in sorted((repo_root/'target').glob('agentic-flink-*.jar'))
            if 'original' not in j.name and 'sources' not in j.name]
    assert jars, 'build the jar: mvn -DskipTests package'
    os.environ['AGENTIC_FLINK_JAR'] = str(jars[-1])
cp = repo_root/'target'/'runtime-classpath.txt'
if not cp.exists():
    subprocess.run(['mvn','-q','dependency:build-classpath',f'-Dmdep.outputFile={cp}',
                    '-Dmdep.includeScope=runtime'], cwd=repo_root, check=True)
extra_jars = [j for j in cp.read_text().strip().split(':') if j]
import agentic_flink as af
af.start_jvm(extra_jars=extra_jars)   # JVM defaults include the Arrow --add-opens Fluss needs
from agentic_flink import jclass
from java.util import HashMap
print('JVM up:', af.is_started())

JVM up: True


## 2. Build a Fluss-backed knowledge base

The `KnowledgeBase` keeps its KNN index in memory but writes the durable copy of every
chunk (id, vector, text, metadata) into a Fluss primary-key table.

In [2]:
OLLAMA_URL = os.environ.get('OLLAMA_URL', 'http://localhost:11434')
FLUSS_BOOTSTRAP = os.environ.get('FLUSS_BOOTSTRAP', 'localhost:9123')
TABLE = os.environ.get('FLUSS_TABLE', 'research_corpus')
ANTHROPIC_KEY = os.environ.get('ANTHROPIC_API_KEY')

def fluss_store(table):
    FlussVS = jclass('org.agentic.flink.storage.vector.FlussVectorStore')
    cfg = HashMap()
    for k, v in {'fluss.bootstrap.servers': FLUSS_BOOTSTRAP, 'fluss.database': 'agentic_flink',
                 'fluss.table': table, 'fluss.buckets': '1',
                 'vector.dimension': '768', 'vector.similarity': 'cosine'}.items():
        cfg.put(k, v)
    s = FlussVS(); s.initialize(cfg); return s

def build_kb(store):
    KB = jclass('org.agentic.flink.rag.KnowledgeBase')
    b = KB.builder().withOllamaEmbedding(OLLAMA_URL, 'nomic-embed-text', 768).withVectorStore(store)
    if ANTHROPIC_KEY:
        b = b.withClaude(ANTHROPIC_KEY)
    return b.build()

kb = build_kb(fluss_store(TABLE))
print('KnowledgeBase backed by Fluss table', repr(TABLE))
print('starting corpus size (hydrated from Fluss):', kb.stats().get('total_vectors'))
print('chat:', 'Claude' if ANTHROPIC_KEY else 'disabled (no ANTHROPIC_API_KEY)')

KnowledgeBase backed by Fluss table 'research_1779905216'
starting corpus size (hydrated from Fluss): 0
chat: disabled (no ANTHROPIC_API_KEY)


## 3. Research pass: scrape targets into Fluss

Set `SCRAPE_URLS` (comma-separated) to your research targets. Each page is scraped,
chunked, embedded with Ollama, and persisted to Fluss.

In [3]:
default_urls = [
    'https://flink.apache.org/what-is-flink/flink-architecture/',
    'https://en.wikipedia.org/wiki/Apache_Flink',
]
urls = [u for u in os.environ.get('SCRAPE_URLS', ','.join(default_urls)).split(',') if u]
for u in urls:
    r = kb.ingestUrl(u)
    print(f'  {u} -> {"%d chunks" % r.chunks if r.ok else "FAILED: "+str(r.error)} ({r.title})')
print('\ncorpus size in Fluss now:', kb.stats().get('total_vectors'))

  http://localhost:8077/flink.html -> 1 chunks (Apache Flink Overview)


  http://localhost:8077/agents.html -> 1 chunks (Agentic Flink)

corpus size in Fluss now: 2


## 4. Durability: a fresh researcher re-hydrates from Fluss

Build a brand-new `KnowledgeBase` pointing at the **same Fluss table**. It replays the
table's changelog to rebuild its search index — the research survives the process, with
no re-scraping.

In [4]:
kb2 = build_kb(fluss_store(TABLE))
print('fresh KB hydrated', kb2.stats().get('total_vectors'), 'chunks from Fluss (no scraping)')
for p in kb2.search('How does Flink recover state after a failure?', 3):
    print(f'  [{p.score:.3f}] {p.title or p.id}: {str(p.text)[:90].strip()}...')

fresh KB hydrated 2 chunks from Fluss (no scraping)
  [0.599] Apache Flink Overview: Apache Flink Apache Flink is an open-source, distributed stream-processing framework for s...
  [0.589] Agentic Flink: Agentic Flink Agentic Flink layers LLM-powered agents on top of Apache Flink. Memory lives...


## 5. Ask the researcher (Claude, grounded in the Fluss corpus)

In [5]:
if not ANTHROPIC_KEY:
    print('⚠ No ANTHROPIC_API_KEY — skipping. Set it and re-run cells 1-2 to enable.')
else:
    ans = kb2.ask('What is Apache Flink and what guarantees does it provide?', 4)
    print('ANSWER:\n', ans.text, '\n\nSOURCES:')
    for p in ans.sources:
        print(f'  - {p.title or p.id} ({p.url})')

⚠ No ANTHROPIC_API_KEY — skipping. Set it and re-run cells 1-2 to enable.


## Done

The corpus is durably stored in Apache Fluss: scraping persists chunks to a Fluss
primary-key table, and any new researcher process rebuilds its index by replaying the
Fluss changelog. Swap `nomic-embed-text` for any Ollama embedding model, or point
`FLUSS_BOOTSTRAP` at a multi-node Fluss cluster — same code.